In [ ]:
# %%
# xyz2.ipynb - cell 1: setup, imports, configuration (noisy detector labels - mapped to binary)

import pandas as pd
import numpy as np
from pathlib import Path
import sys
import random
import joblib
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Optional, Any, Tuple, Dict

# --- project root ---
# sets up the project's root path for local module imports
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# --- import from our csi_preprocessing module ---
from csi_preprocessing import (
    load_app_config,
    get_raw_data_path,
    get_config_csi_metadata_cols,
    remove_known_dead_csi_columns
)
print("necessary functions from csi_preprocessing imported.")

# --- load global configuration ---
print("\n--- loading global configuration ---")
try:
    config = load_app_config(PROJECT_ROOT)
    print("global configuration loaded successfully.")
    METADATA_COLS_FROM_CONFIG = get_config_csi_metadata_cols(config)
    OLD_PIPELINE_COLUMNS_TO_REMOVE = ["conf", "det ts", "csi ts"]
    TARGET_LABEL_COL = config.get("target_label_column", "people")
except Exception as e:
    print(f"error loading global configuration: {e}")
    raise

# --- choose labeling strategy for this notebook ---
# this flag is the key difference from the fgt notebook
# it instructs the pipeline to use the raw detector output for labels
USE_GROUND_TRUTH_LABELS = False
# --- ----------------------------------------- ---

EXPERIMENT_NAME = "Noisy_Detector_Labels_Old_Pipeline_Rep"
PIPELINE_CONFIG_KEY = "detector_labels"

if PIPELINE_CONFIG_KEY not in config.get("experiment_pipelines", {}):
    print(f"warning: pipeline configuration for '{PIPELINE_CONFIG_KEY}' not found in config.json. using default suffix.")
    EXPERIMENT_SUFFIX = "noisy_detector"
else:
    EXPERIMENT_SUFFIX = config["experiment_pipelines"][PIPELINE_CONFIG_KEY]["pipeline_suffix"]

# --- define output paths for this experiment (xyz2) ---
# creates unique directories for this rdl experiment
BASE_OUTPUT_DIR_XYZ2 = PROJECT_ROOT / f"experiment_outputs_{EXPERIMENT_SUFFIX}"
XYZ2_CLEAN_DIR = BASE_OUTPUT_DIR_XYZ2 / "data_clean"
XYZ2_ARTIFACTS_DIR = BASE_OUTPUT_DIR_XYZ2 / "artifacts"
XYZ2_WINDOWED_DATA_DIR = BASE_OUTPUT_DIR_XYZ2 / "windowed_data"
XYZ2_TRAIN_WINDOW_DIR = XYZ2_WINDOWED_DATA_DIR / "train"
XYZ2_VAL_WINDOW_DIR = XYZ2_WINDOWED_DATA_DIR / "val"
XYZ2_TEST_WINDOW_DIR = XYZ2_WINDOWED_DATA_DIR / "test"
XYZ2_MODELS_DIR = PROJECT_ROOT / config["models_storage_base_dir"] / f"csi_detector_{EXPERIMENT_SUFFIX}"
XYZ2_PLOTS_DIR = PROJECT_ROOT / config["plots_base_dir"] / f"training_plots_{EXPERIMENT_SUFFIX}"

for dir_path in [XYZ2_CLEAN_DIR, XYZ2_ARTIFACTS_DIR, XYZ2_WINDOWED_DATA_DIR,
                 XYZ2_TRAIN_WINDOW_DIR, XYZ2_VAL_WINDOW_DIR, XYZ2_TEST_WINDOW_DIR,
                 XYZ2_MODELS_DIR, XYZ2_PLOTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"\n--- paths for experiment: {EXPERIMENT_NAME} ---")
print(f"clean data dir: {XYZ2_CLEAN_DIR}")
print(f"artifacts (scaler, pca) dir: {XYZ2_ARTIFACTS_DIR}")
print(f"windowed data dir: {XYZ2_WINDOWED_DATA_DIR}")
print(f"trained models dir: {XYZ2_MODELS_DIR}")
print(f"training plots dir: {XYZ2_PLOTS_DIR}")

# sets parameters for the entire pipeline, identical to the fgt notebook
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

DWT_BEFORE_PCA = True
DWT_WAVELET_OLD = 'db4'
DWT_LEVEL_OLD = 4
PCA_N_COMPONENTS_OLD = 0.99
PCA_SVD_SOLVER_OLD = "full"
PCA_RANDOM_STATE_OLD = SEED
WIN_SIZE_XYZ2 = 16
STEP_SIZE_XYZ2 = 8
MLP_EPOCHS_XYZ2 = 60
MLP_BATCH_SIZE_XYZ2 = 512
MLP_LEARNING_RATE_XYZ2 = 3e-4
sns.set_theme(style="whitegrid", palette="pastel")
PLOT_DPI = 200

print("\n✔ cell 1: configuration for xyz2.ipynb (noisy detector labels) done.")
print(f"dwt_before_pca set to: {DWT_BEFORE_PCA}")
print(f"win_size_xyz2 set to: {WIN_SIZE_XYZ2}, step_size_xyz2 set to: {STEP_SIZE_XYZ2}")
print(f"use_ground_truth_labels set to: {USE_GROUND_TRUTH_LABELS} (meaning detector labels will be used and mapped)")

In [ ]:
# %%
# xyz2.ipynb - cell 2: initial data cleaning (using detector labels)

print(f"\n--- starting data cleaning for: {EXPERIMENT_NAME} ---")
files_cleaned_count_xyz2 = 0
for category_key, sub_categories in config["raw_data_categories"].items():
    true_label_for_category = -1
    if "0_pers" in category_key.lower(): true_label_for_category = 0
    elif "1_pers" in category_key.lower(): true_label_for_category = 1

    for sub_category_key, _ in sub_categories.items():
        current_raw_data_dir = get_raw_data_path(PROJECT_ROOT, config, category_key, sub_category_key)
        output_clean_subdir_xyz2 = XYZ2_CLEAN_DIR / category_key / sub_category_key
        output_clean_subdir_xyz2.mkdir(parents=True, exist_ok=True)

        if not current_raw_data_dir.exists():
            print(f"warning: raw data directory not found: {current_raw_data_dir}")
            continue

        for raw_csv_file_path in sorted(list(current_raw_data_dir.glob('*.csv'))):
            try:
                df = pd.read_csv(raw_csv_file_path)
                df_cleaned = df.drop(columns=[c for c in OLD_PIPELINE_COLUMNS_TO_REMOVE if c in df.columns], errors='ignore')

                # because use_ground_truth_labels is false, this 'if' is skipped
                if USE_GROUND_TRUTH_LABELS:
                    if true_label_for_category != -1:
                        df_cleaned[TARGET_LABEL_COL] = true_label_for_category
                    else:
                        if TARGET_LABEL_COL not in df_cleaned.columns and TARGET_LABEL_COL in df.columns:
                            df_cleaned[TARGET_LABEL_COL] = df[TARGET_LABEL_COL]
                # this 'else' block executes, preserving the original 'people' column from the csv
                else:
                    if TARGET_LABEL_COL not in df_cleaned.columns and TARGET_LABEL_COL in df.columns:
                         df_cleaned[TARGET_LABEL_COL] = df[TARGET_LABEL_COL]
                    elif TARGET_LABEL_COL not in df_cleaned.columns and TARGET_LABEL_COL not in df.columns:
                        print(f"error: target label column '{TARGET_LABEL_COL}' not found in {raw_csv_file_path.name}. skipping.")
                        continue

                # removes the 12 dead subcarriers as before
                df_cleaned = remove_known_dead_csi_columns(df_cleaned.copy())

                output_file_path = output_clean_subdir_xyz2 / raw_csv_file_path.name
                df_cleaned.to_csv(output_file_path, index=False)
                files_cleaned_count_xyz2 += 1
            except Exception as e:
                print(f"error cleaning file {raw_csv_file_path.name}: {e}")
                import traceback; traceback.print_exc()

print(f"\n✔ cell 2: data cleaning for {EXPERIMENT_NAME} complete. {files_cleaned_count_xyz2} files cleaned into '{XYZ2_CLEAN_DIR}'.")

In [ ]:
# %%
# xyz2.ipynb - cell 3: define "old pipeline" processing helper functions (identical to abc1.ipynb's cell 3)

# loads x and y from a cleaned csv; y contains the raw detector labels (0, 1, 2...)
def load_cleaned_xyz2_csv(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)
    if TARGET_LABEL_COL not in df.columns:
        raise ValueError(f"target label column '{TARGET_LABEL_COL}' not found in {path}")
    X = df.drop(columns=[TARGET_LABEL_COL], errors='ignore').values.astype(np.float32)
    y = df[TARGET_LABEL_COL].values.astype(np.int64)
    return X, y

# applies dwt denoising, identical to the fgt notebook
def dwt_denoise_matrix_xyz2(X: np.ndarray, wavelet: str = DWT_WAVELET_OLD, level: int = DWT_LEVEL_OLD) -> np.ndarray:
    n_rows, n_features = X.shape; Xd = np.zeros_like(X)
    if n_rows == 0: return Xd
    wavelet_obj = pywt.Wavelet(wavelet)
    for i in range(n_features):
        col = X[:, i]; current_max_level = pywt.dwt_max_level(len(col), wavelet_obj)
        actual_level = min(level, current_max_level)
        if actual_level < 1: Xd[:, i] = col; continue
        coeffs = pywt.wavedec(col, wavelet, level=actual_level)
        detail_coeffs_finest = coeffs[-1]; sigma = 0
        if len(detail_coeffs_finest) > 0: sigma = np.median(np.abs(detail_coeffs_finest - np.median(detail_coeffs_finest))) / 0.6745
        thresh = 0.0
        if sigma > 0 and n_rows > 1: thresh = sigma * np.sqrt(2 * np.log(n_rows))
        elif sigma > 0 and n_rows == 1: thresh = sigma
        coeffs_thresholded = [coeffs[0]] + [pywt.threshold(c, thresh, mode="soft") for c in coeffs[1:]]
        reconstructed_col = pywt.waverec(coeffs_thresholded, wavelet)
        if len(reconstructed_col) > n_rows: Xd[:, i] = reconstructed_col[:n_rows]
        elif len(reconstructed_col) < n_rows and len(reconstructed_col) > 0:
            Xd[:, i] = np.concatenate([reconstructed_col, np.full(n_rows - len(reconstructed_col), reconstructed_col[-1])])
        elif len(reconstructed_col) == 0 and n_rows > 0: Xd[:, i] = np.zeros(n_rows)
        else: Xd[:, i] = reconstructed_col
    return Xd

# applies a fitted pca model, identical to the fgt notebook
def apply_pca_xyz2(X: np.ndarray, pca_model: PCA) -> np.ndarray:
    if not hasattr(pca_model, 'n_components_') or pca_model.n_components_ == 0: return X
    if X.shape[1] != pca_model.n_features_in_:
        return X
    return pca_model.transform(X)

# creates 3d windows and assigns a binary label to each
def window_data_xyz2_3d(X: np.ndarray, y: np.ndarray, window_size: int, step_size: int, use_gt_labels_in_windowing: bool) -> Tuple[np.ndarray, np.ndarray]:
    segments, labels = [], []
    N, num_features = X.shape
    if N < window_size or num_features == 0: return np.array([]).reshape(0, window_size, num_features), np.array([])
    for start in range(0, N - window_size + 1, step_size):
        segments.append(X[start : start + window_size])
        label_segment = y[start : start + window_size]
        current_window_label = 0
        if len(label_segment) > 0:
            # this 'if' is skipped since use_gt_labels_in_windowing is false
            if use_gt_labels_in_windowing:
                current_window_label = int(np.mean(label_segment) > 0.5)
            # this 'else' block maps the raw detector labels to binary
            # if any frame in the window has a person count > 0, the window label is 1
            else:
                if any(val != 0 for val in label_segment):
                    current_window_label = 1
                else: current_window_label = 0
        labels.append(current_window_label)
    if not segments: return np.array([]).reshape(0, window_size, num_features), np.array([])
    return np.stack(segments).astype(np.float32), np.array(labels).astype(np.int64)

print("✔ cell 3: helper functions for xyz2.ipynb defined.")

In [ ]:
# %%
# xyz2.ipynb - cell 4: stratified file splitting, scaler & pca fitting

print(f"\n--- starting stratified file splitting, scaler & pca fitting for: {EXPERIMENT_NAME} ---")

# 1. groups cleaned csv files by their sub-location for stratified splitting
files_by_sublocation_xyz2 = {}
for category_key, sub_categories_map in config["raw_data_categories"].items():
    for sub_category_key, _ in sub_categories_map.items():
        sub_loc_key = f"{category_key}_{sub_category_key}"
        current_clean_sub_dir = XYZ2_CLEAN_DIR / category_key / sub_category_key
        if current_clean_sub_dir.exists():
            files = sorted(list(current_clean_sub_dir.glob("*.csv")))
            if files: files_by_sublocation_xyz2[sub_loc_key] = files
print("\nfiles collected per sub-location for xyz2:")
for sub_loc, f_list in files_by_sublocation_xyz2.items(): print(f"  {sub_loc}: {len(f_list)} files")

# 2. performs a stratified split on the files to create train and test sets
train_files_xyz2, test_files_xyz2 = [], []
np.random.seed(SEED)
for sub_loc_key, files_in_sub_loc in files_by_sublocation_xyz2.items():
    num_files_in_sub_loc = len(files_in_sub_loc); n_test_for_sub_loc = 0
    if num_files_in_sub_loc > 1 : n_test_for_sub_loc = max(1, int(num_files_in_sub_loc * 0.20))
    if num_files_in_sub_loc - n_test_for_sub_loc < 1 and num_files_in_sub_loc > 1: n_test_for_sub_loc = num_files_in_sub_loc - 1
    if num_files_in_sub_loc == 1: n_test_for_sub_loc = 0
    chosen_test = list(np.random.choice(files_in_sub_loc, n_test_for_sub_loc, replace=False)) if n_test_for_sub_loc > 0 else []
    chosen_train = [f for f in files_in_sub_loc if f not in chosen_test]
    test_files_xyz2.extend(chosen_test); train_files_xyz2.extend(chosen_train)

print(f"\ntotal cleaned files for xyz2: {sum(len(v) for v in files_by_sublocation_xyz2.values())}")
print(f"xyz2 train files: {len(train_files_xyz2)}")
print(f"xyz2 test files: {len(test_files_xyz2)}")

# 3. stacks data from only the training files to fit the scaler and pca model
X_stack_xyz2_list, y_stack_xyz2_detector_labels = [], []
for p_path in train_files_xyz2:
    X_f, y_f_detector = load_cleaned_xyz2_csv(p_path)
    if X_f.shape[0] > 0: X_stack_xyz2_list.append(X_f); y_stack_xyz2_detector_labels.append(y_f_detector)
if not X_stack_xyz2_list: raise ValueError("no data in training files for xyz2.")
X_stack_xyz2 = np.vstack(X_stack_xyz2_list)
y_stack_xyz2_original_labels = np.concatenate(y_stack_xyz2_detector_labels)
print(f"xyz2 x_stack shape (for scaler/pca): {X_stack_xyz2.shape}")
unique_orig_labels_train, counts_orig_labels_train = np.unique(y_stack_xyz2_original_labels, return_counts=True)
print(f"distribution of original detector labels in xyz2 training stack: {dict(zip(unique_orig_labels_train, counts_orig_labels_train))}")

# 4. fits the standardscaler on the training stack and saves it
scaler_xyz2 = StandardScaler().fit(X_stack_xyz2)
scaler_save_path_xyz2 = XYZ2_ARTIFACTS_DIR / f"scaler_xyz2_{EXPERIMENT_SUFFIX}.joblib"
joblib.dump(scaler_xyz2, scaler_save_path_xyz2)
print(f"xyz2 scaler saved: {scaler_save_path_xyz2}")

# 5. scales the data, applies dwt, then fits the pca model and saves it
Xn_xyz2 = scaler_xyz2.transform(X_stack_xyz2)
X_for_pca_fit_xyz2 = Xn_xyz2
if DWT_BEFORE_PCA:
    print("applying dwt before pca to xyz2 training stack...")
    X_for_pca_fit_xyz2 = dwt_denoise_matrix_xyz2(Xn_xyz2.copy())
else:
    print("skipping dwt before pca for xyz2 (dwt applied later to pcs per file).")
if X_for_pca_fit_xyz2.shape[1] == 0: raise ValueError("no features for pca in xyz2.")
pca_xyz2 = PCA(n_components=PCA_N_COMPONENTS_OLD, svd_solver=PCA_SVD_SOLVER_OLD, random_state=PCA_RANDOM_STATE_OLD)
pca_xyz2.fit(X_for_pca_fit_xyz2)
pca_save_path_xyz2 = XYZ2_ARTIFACTS_DIR / f"pca_xyz2_{EXPERIMENT_SUFFIX}.joblib"
joblib.dump(pca_xyz2, pca_save_path_xyz2)
print(f"xyz2 pca model saved: {pca_save_path_xyz2}")
n_pca_comp_xyz2 = getattr(pca_xyz2, 'n_components_', 0)
print(f"  pca components fitted: {n_pca_comp_xyz2}")
if n_pca_comp_xyz2 > 0: print(f"  explained variance: {np.sum(pca_xyz2.explained_variance_ratio_):.4f}")

print(f"\n✔ cell 4: scaler and pca for {EXPERIMENT_NAME} fitted and saved.")

In [ ]:
# %%
# xyz2.ipynb - cell 5: windowing and data processing for each split

print(f"\n--- starting windowing for: {EXPERIMENT_NAME} ---")

scaler_loaded_xyz2 = joblib.load(XYZ2_ARTIFACTS_DIR / f"scaler_xyz2_{EXPERIMENT_SUFFIX}.joblib")
pca_loaded_xyz2 = joblib.load(XYZ2_ARTIFACTS_DIR / f"pca_xyz2_{EXPERIMENT_SUFFIX}.joblib")
print(f"reloaded xyz2 scaler and pca model for {EXPERIMENT_SUFFIX}.")
print(f"using fixed win_size_xyz2={WIN_SIZE_XYZ2}, step_size_xyz2={STEP_SIZE_XYZ2}")

apply_dwt_after_pca_flag_xyz2 = not DWT_BEFORE_PCA

# processes a list of files using the fitted scaler and pca models
def process_data_split_xyz2(files_list: List[Path],
                            scaler: StandardScaler, pca_model: PCA,
                            win_sz: int, step_sz: int,
                            dwt_after_pca: bool, dwt_before_pca: bool,
                            use_gt_labels_in_windowing: bool):
    print(f"processing {len(files_list)} files for windowing...")
    X_all_win, y_all_win = [], []
    for p_path in files_list:
        X_f, y_f_orig_detector = load_cleaned_xyz2_csv(p_path)
        if X_f.shape[0] < win_sz: continue
        Xn = scaler.transform(X_f)
        X_pre_pca = Xn
        if dwt_before_pca: X_pre_pca = dwt_denoise_matrix_xyz2(Xn.copy())
        Xp = apply_pca_xyz2(X_pre_pca, pca_model)
        X_final = Xp
        if dwt_after_pca: X_final = dwt_denoise_matrix_xyz2(Xp.copy())

        # this function call is where the raw detector labels get mapped to binary window labels
        Xw, yw = window_data_xyz2_3d(X_final, y_f_orig_detector,
                                     win_sz, step_sz,
                                     use_gt_labels_in_windowing=USE_GROUND_TRUTH_LABELS)
        if Xw.size > 0: X_all_win.append(Xw); y_all_win.append(yw)

    if not X_all_win:
        n_feat = getattr(pca_model, 'n_components_', X_f.shape[1] if 'X_f' in locals() and X_f is not None else 1)
        return np.array([]).reshape(0, win_sz, n_feat if n_feat > 0 else 1), np.array([])
    return np.concatenate(X_all_win, axis=0), np.concatenate(y_all_win, axis=0)

# splits training files into a train and validation set
stratify_labels_for_val_split_xyz2 = [1 if "1_pers" in str(p).lower() else 0 for p in train_files_xyz2]
can_stratify_val_xyz2 = len(np.unique(stratify_labels_for_val_split_xyz2)) > 1 and \
                   all(np.sum(np.array(stratify_labels_for_val_split_xyz2) == lbl) >= 2
                       for lbl in np.unique(stratify_labels_for_val_split_xyz2))
val_split_size_xyz2 = 0.15

if len(train_files_xyz2) * val_split_size_xyz2 < 1 or not can_stratify_val_xyz2 :
    train_sub_files_xyz2 = train_files_xyz2; val_files_xyz2 = []
else:
    try:
        train_sub_files_xyz2, val_files_xyz2 = train_test_split(
            train_files_xyz2, test_size=val_split_size_xyz2,
            stratify=stratify_labels_for_val_split_xyz2, random_state=SEED, shuffle=True
        )
    except ValueError:
        train_sub_files_xyz2, val_files_xyz2 = train_test_split(train_files_xyz2, test_size=val_split_size_xyz2, random_state=SEED, shuffle=True)

print(f"xyz2 final training files: {len(train_sub_files_xyz2)}")
print(f"xyz2 final validation files: {len(val_files_xyz2)}")

# processes the train, validation, and test file sets separately
X_train_xyz2, y_train_xyz2 = process_data_split_xyz2(
    train_sub_files_xyz2, scaler_loaded_xyz2, pca_loaded_xyz2,
    WIN_SIZE_XYZ2, STEP_SIZE_XYZ2, apply_dwt_after_pca_flag_xyz2, DWT_BEFORE_PCA,
    use_gt_labels_in_windowing=USE_GROUND_TRUTH_LABELS
)
if val_files_xyz2:
    X_val_xyz2, y_val_xyz2 = process_data_split_xyz2(
        val_files_xyz2, scaler_loaded_xyz2, pca_loaded_xyz2,
        WIN_SIZE_XYZ2, STEP_SIZE_XYZ2, apply_dwt_after_pca_flag_xyz2, DWT_BEFORE_PCA,
        use_gt_labels_in_windowing=USE_GROUND_TRUTH_LABELS
    )
else:
    n_feat_val_xyz2 = getattr(pca_loaded_xyz2, 'n_components_', X_stack_xyz2.shape[1] if 'X_stack_xyz2' in locals() else 1)
    X_val_xyz2, y_val_xyz2 = np.array([]).reshape(0, WIN_SIZE_XYZ2, n_feat_val_xyz2 if n_feat_val_xyz2 > 0 else 1), np.array([])

X_test_xyz2, y_test_xyz2 = process_data_split_xyz2(
    test_files_xyz2, scaler_loaded_xyz2, pca_loaded_xyz2,
    WIN_SIZE_XYZ2, STEP_SIZE_XYZ2, apply_dwt_after_pca_flag_xyz2, DWT_BEFORE_PCA,
    use_gt_labels_in_windowing=USE_GROUND_TRUTH_LABELS
)

# saves the final windowed data arrays to disk
np.save(XYZ2_TRAIN_WINDOW_DIR / "X_train_windows.npy", X_train_xyz2)
np.save(XYZ2_TRAIN_WINDOW_DIR / "y_train_labels.npy", y_train_xyz2)
np.save(XYZ2_VAL_WINDOW_DIR / "X_val_windows.npy", X_val_xyz2)
np.save(XYZ2_VAL_WINDOW_DIR / "y_val_labels.npy", y_val_xyz2)
np.save(XYZ2_TEST_WINDOW_DIR / "X_test_windows.npy", X_test_xyz2)
np.save(XYZ2_TEST_WINDOW_DIR / "y_test_labels.npy", y_test_xyz2)
print(f"final windowed data for {EXPERIMENT_NAME} saved to {XYZ2_WINDOWED_DATA_DIR}")

print(f"\nxyz2 processed train data -> x:{X_train_xyz2.shape}, y:{y_train_xyz2.shape}")
if y_train_xyz2.size > 0: print(f"  unique y_train_xyz2 labels (mapped to binary): {np.unique(y_train_xyz2)}")

print(f"dwt was applied {'before' if DWT_BEFORE_PCA else 'after'} pca for this xyz2 run.")
print(f"labels used for windowing: {'ground truth' if USE_GROUND_TRUTH_LABELS else 'detector mapped to binary'}")
print("\n✔ cell 5: windowing complete for xyz2.ipynb.")

In [ ]:
# %%
# xyz2.ipynb - cell 6: mlp training and evaluation

print(f"\n--- starting mlp training & evaluation for: {EXPERIMENT_NAME} ---")

if 'X_train_xyz2' not in locals() or X_train_xyz2.size == 0 or \
   'X_val_xyz2' not in locals() or \
   'X_test_xyz2' not in locals() or X_test_xyz2.size == 0:
    print(f"error: training, validation, or test data for {EXPERIMENT_NAME} is empty or not defined. skipping mlp training.")
else:
    # flattens the 3d windowed data to 2d for the mlp input
    def flatten_xyz2(X_3d: np.ndarray) -> np.ndarray:
        if X_3d.ndim == 3:
            n_s, n_t, n_f = X_3d.shape
            if n_f == 0 and n_s > 0: return np.array([]).reshape(n_s, 0)
            return X_3d.reshape(n_s, n_t * n_f).astype(np.float32)
        return X_3d.astype(np.float32)

    X_tr_flat_xyz2 = flatten_xyz2(X_train_xyz2)
    X_va_flat_xyz2 = flatten_xyz2(X_val_xyz2) if X_val_xyz2.size > 0 else np.array([]).reshape(0, X_tr_flat_xyz2.shape[1] if X_tr_flat_xyz2.size > 0 else 0)
    X_test_flat_xyz2 = flatten_xyz2(X_test_xyz2)

    if X_tr_flat_xyz2.shape[0] > 0 and X_tr_flat_xyz2.shape[1] == 0 and X_train_xyz2.shape[0] > 0 and X_train_xyz2.shape[2] > 0 :
        print("error: input dimension for mlp is 0. cannot proceed.")
    elif X_tr_flat_xyz2.shape[0] == 0 :
        print("error: training data for xyz2 is empty. cannot proceed.")
    else:
        input_dim_xyz2 = X_tr_flat_xyz2.shape[1]
        print(f"xyz2 mlp input dimension: {input_dim_xyz2}")

        # calculates class weights to handle data imbalance
        cw_dict_xyz2 = None
        if len(y_train_xyz2) > 0 and len(np.unique(y_train_xyz2)) > 1:
            try:
                cw_vals_xyz2 = class_weight.compute_class_weight(class_weight="balanced", classes=np.unique(y_train_xyz2), y=y_train_xyz2)
                cw_dict_xyz2 = {int(c): float(w) for c, w in zip(np.unique(y_train_xyz2), cw_vals_xyz2)}
            except ValueError: cw_dict_xyz2 = None
        print(f"xyz2 class weights: {cw_dict_xyz2}")

        # defines the mlp architecture, identical to the one in the paper
        def build_mlp_xyz2(dim, name_suffix=""):
            inp = layers.Input(shape=(dim,))
            x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(inp)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.3)(x)
            x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.3)(x)
            x = layers.Dense(64,  kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.2)(x)
            x = layers.Dense(32,  kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
            out = layers.Dense(1, activation="sigmoid")(x)
            model = tf.keras.Model(inp, out, name=f"MLP_XYZ2_{EXPERIMENT_SUFFIX}{name_suffix}")
            model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LEARNING_RATE_XYZ2), loss="binary_crossentropy", metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
            return model

        model_suffix_detail_xyz2 = f"_W{WIN_SIZE_XYZ2}_S{STEP_SIZE_XYZ2}_DWT_{'BeforePCA' if DWT_BEFORE_PCA else 'AfterPCA'}"
        mlp_xyz2_model_name = f"MLP_XYZ2{model_suffix_detail_xyz2}"

        mlp_xyz2 = build_mlp_xyz2(input_dim_xyz2, name_suffix=model_suffix_detail_xyz2)
        print(f"\n--- training {mlp_xyz2_model_name} ---")
        mlp_xyz2.summary()

        model_checkpoint_path_xyz2 = str(XYZ2_MODELS_DIR / mlp_xyz2_model_name / "best_model.keras")
        (XYZ2_MODELS_DIR / mlp_xyz2_model_name).mkdir(parents=True, exist_ok=True)

        validation_data_fit_xyz2 = None
        if X_va_flat_xyz2.size > 0 and y_val_xyz2.size > 0:
            validation_data_fit_xyz2 = (X_va_flat_xyz2, y_val_xyz2.astype(np.int32))

        # trains the model with callbacks for early stopping and saving the best version
        history_xyz2 = mlp_xyz2.fit(
            X_tr_flat_xyz2, y_train_xyz2.astype(np.int32),
            epochs=MLP_EPOCHS_XYZ2, batch_size=MLP_BATCH_SIZE_XYZ2,
            validation_data=validation_data_fit_xyz2,
            class_weight=cw_dict_xyz2,
            callbacks=[
                callbacks.ReduceLROnPlateau(monitor='val_loss' if validation_data_fit_xyz2 else 'loss', patience=10, factor=0.5, min_lr=1e-6, verbose=1),
                callbacks.EarlyStopping(monitor='val_loss' if validation_data_fit_xyz2 else 'loss', patience=15, restore_best_weights=True, verbose=1),
                callbacks.ModelCheckpoint(filepath=model_checkpoint_path_xyz2, save_best_only=True,
                                          monitor='val_accuracy' if validation_data_fit_xyz2 else 'accuracy',
                                          mode='max', verbose=1)
            ], verbose=2
        )

        if Path(model_checkpoint_path_xyz2).exists(): mlp_xyz2.load_weights(model_checkpoint_path_xyz2)

        # evaluates the final model on the unseen test set
        print(f"\n--- evaluating {mlp_xyz2_model_name} on xyz2 test set ---")
        eval_results_xyz2 = mlp_xyz2.evaluate(X_test_flat_xyz2, y_test_xyz2.astype(np.int32), verbose=0)
        y_prob_xyz2 = mlp_xyz2.predict(X_test_flat_xyz2).ravel()
        y_pred_xyz2 = (y_prob_xyz2 > 0.5).astype(int)

        print(f"\n{mlp_xyz2_model_name} - test results ({EXPERIMENT_NAME}):")
        print(f"  loss: {eval_results_xyz2[0]:.4f}"); print(f"  accuracy: {eval_results_xyz2[1]:.4f}")
        print(f"  precision (keras): {eval_results_xyz2[2]:.4f}"); print(f"  recall (keras): {eval_results_xyz2[3]:.4f}")
        f1_xyz2 = f1_score(y_test_xyz2,y_pred_xyz2,pos_label=1,average='binary',zero_division=0)
        prec_sk_xyz2 = precision_score(y_test_xyz2,y_pred_xyz2,pos_label=1,average='binary',zero_division=0)
        rec_sk_xyz2 = recall_score(y_test_xyz2,y_pred_xyz2,pos_label=1,average='binary',zero_division=0)
        print(f"  f1-score (sklearn class 1): {f1_xyz2:.4f}")
        print(f"  precision (sklearn class 1): {prec_sk_xyz2:.4f}")
        print(f"  recall (sklearn class 1): {rec_sk_xyz2:.4f}")
        roc_auc_xyz2 = float('nan')
        if len(np.unique(y_test_xyz2)) > 1:
            try: roc_auc_xyz2 = roc_auc_score(y_test_xyz2, y_prob_xyz2)
            except ValueError: pass
        if np.isnan(roc_auc_xyz2): print("  roc auc: could not be calculated")
        else: print(f"  roc auc: {roc_auc_xyz2:.4f}")

        mlp_xyz2.save(XYZ2_MODELS_DIR / f"{mlp_xyz2_model_name}_final.keras")
        print(f"✔ {mlp_xyz2_model_name} (final) saved to {XYZ2_MODELS_DIR}.")

        # generates and saves evaluation plots
        xyz2_plot_save_dir = XYZ2_PLOTS_DIR / mlp_xyz2_model_name
        xyz2_plot_save_dir.mkdir(parents=True, exist_ok=True)

        def plot_training_history_abc1(history, model_name_str, save_dir):
            plt.figure(figsize=(14, 6)); plt.subplot(1, 2, 1)
            if 'accuracy' in history.history: plt.plot(history.history['accuracy'], label='train acc', marker='.')
            if 'val_accuracy' in history.history: plt.plot(history.history['val_accuracy'], label='val acc', marker='.')
            plt.title('accuracy', fontsize=14); plt.xlabel('epoch'); plt.ylabel('accuracy')
            if 'accuracy' in history.history or 'val_accuracy' in history.history: plt.legend()
            plt.grid(True, alpha=0.7); plt.subplot(1, 2, 2)
            if 'loss' in history.history: plt.plot(history.history['loss'], label='train loss', marker='.')
            if 'val_loss' in history.history: plt.plot(history.history['val_loss'], label='val loss', marker='.')
            plt.title('loss', fontsize=14); plt.xlabel('epoch'); plt.ylabel('loss')
            if 'loss' in history.history or 'val_loss' in history.history: plt.legend()
            plt.grid(True, alpha=0.7); plt.suptitle(f"training: {model_name_str}", fontsize=16, fontweight='bold')
            plt.tight_layout(rect=[0,0.03,1,0.95]); plt.savefig(save_dir/"th.png",dpi=PLOT_DPI); plt.close(); print(f"plot: {save_dir/'th.png'}")
        def plot_confusion_matrix_abc1(y_true, y_pred, model_name_str, class_names, save_dir):
            cm=confusion_matrix(y_true,y_pred); plt.figure(figsize=(7,6))
            sns.heatmap(cm,annot=True,fmt='d',cmap='blues',xticklabels=class_names,yticklabels=class_names,annot_kws={"size":16},cbar=False)
            plt.title(f'{model_name_str} - cm',fontsize=16,fontweight='bold');plt.xlabel('predicted');plt.ylabel('actual');plt.tight_layout()
            plt.savefig(save_dir/"cm.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'cm.png'}")
        def plot_roc_curve_abc1(y_true, y_prob, model_name_str, auc, save_dir):
            fpr,tpr,_=roc_curve(y_true,y_prob);plt.figure(figsize=(8,7))
            plt.plot(fpr,tpr,label=f'auc = {auc:.4f}',color='darkorange',lw=2.5);plt.plot([0,1],[0,1],'k--',lw=2)
            plt.xlim([-0.01,1.0]);plt.ylim([0.0,1.01]);plt.xlabel('fpr');plt.ylabel('tpr')
            plt.title(f'{model_name_str} - roc',fontsize=16,fontweight='bold');plt.legend(loc="lower right");plt.grid(True,alpha=0.7);plt.tight_layout()
            plt.savefig(save_dir/"roc.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'roc.png'}")
        def plot_pr_curve_abc1(y_true, y_prob, model_name_str, save_dir):
            prec,rec,_=precision_recall_curve(y_true,y_prob);plt.figure(figsize=(8,7))
            plt.plot(rec,prec,label='pr curve',color='mediumblue',lw=2.5)
            plt.xlabel('recall');plt.ylabel('precision');plt.ylim([0.0,1.01]);plt.xlim([0.0,1.0])
            plt.title(f'{model_name_str} - pr curve',fontsize=16,fontweight='bold');plt.grid(True,alpha=0.7);plt.tight_layout()
            plt.savefig(save_dir/"pr_curve.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'pr_curve.png'}")

        plot_training_history_abc1(history_xyz2, mlp_xyz2_model_name, xyz2_plot_save_dir)
        plot_confusion_matrix_abc1(y_test_xyz2, y_pred_xyz2, mlp_xyz2_model_name, ["no person (0)", "person (1)"], xyz2_plot_save_dir)
        if not np.isnan(roc_auc_xyz2): plot_roc_curve_abc1(y_test_xyz2, y_prob_xyz2, mlp_xyz2_model_name, roc_auc_xyz2, xyz2_plot_save_dir)
        plot_pr_curve_abc1(y_test_xyz2, y_prob_xyz2, mlp_xyz2_model_name, xyz2_plot_save_dir)

        print(f"\n✔ cell 6 for xyz2: {mlp_xyz2_model_name} training and evaluation complete.")